In [ ]:
from abc import ABC, abstractmethod
from pathlib import Path
import rdata
import pandas as pd
import os

In [2]:
class RDataLoadError(Exception): 
    def __init__(self, msg):
        self._msg = msg

    def __str__(self):
        return self._msg

class RDataLoadStrategy(ABC):
    @abstractmethod
    def load(self, path: Path, logical_name: str):
        pass


class RdaLoadStrategy(RDataLoadStrategy):
    def load(self, path: Path, logical_name: str):
        parsed = rdata.parser.parse_file(path)
        converted = rdata.conversion.convert(parsed)

        # .rda は dict で返る: {"campaigns": DataFrame} のような形
        return converted[logical_name]


class RdsLoadStrategy(RDataLoadStrategy):
    def load(self, path: Path, logical_name: str):
        parsed = rdata.parser.parse_file(path)
        converted = rdata.conversion.convert(parsed)

        # .rds はオブジェクト自体、今回なら DataFrame が返る
        return converted


class RDataLoader:
    def __init__(self, strategy: RDataLoadStrategy):
        self._strategy = strategy

    def load(self, base_path: Path, logical_name: str, file_type: str):
        strategy = self._strategy
        file_path = base_path / f"{logical_name}.{file_type}"
        return strategy.load(file_path, logical_name)


In [3]:
path_project_base = Path("/loc0/bigbrother/repositories/0015_zenn/workspace")
path_raw = path_project_base / "shared/data/00_raw/completejourney/rdata"

In [4]:

data_names = {
    "campaign_descriptions": "rda", 
    "campaigns": "rda", 
    "coupon_redemptions": "rda", 
    "coupons": "rda", 
    "demographics": "rda", 
    "products": "rda", 
    # "transactions": "rds", 
    # "promotions": "rds"
}

dfs = {}
for name, file_type in data_names.items():
    print(name, file_type)
    if file_type=="rda": 
        loader = RDataLoader(RdaLoadStrategy())
    elif file_type=="rds": 
        loader = RDataLoader(RdsLoadStrategy())
    else: 
        raise RDataLoadError("拡張子は rda または rds を指定してください")


    dfs[name] = loader.load(path_raw, name, file_type)
    print("Done\n")



campaign_descriptions rda
Done

campaigns rda


/loc0/bigbrother/repositories/0015_zenn/workspace/.venv/lib/python3.13/site-packages/rdata/conversion/_conversion.py:900: UserWarning: Missing constructor for R class "Date". The underlying R object is returned instead.
  warnings.warn(
/loc0/bigbrother/repositories/0015_zenn/workspace/.venv/lib/python3.13/site-packages/rdata/conversion/_conversion.py:900: UserWarning: Missing constructor for R class "spec_tbl_df". The constructor for class "tbl_df" will be used instead.
  warnings.warn(
/loc0/bigbrother/repositories/0015_zenn/workspace/.venv/lib/python3.13/site-packages/rdata/conversion/_conversion.py:900: UserWarning: Missing constructor for R class "tbl_df". The constructor for class "tbl" will be used instead.
  warnings.warn(
/loc0/bigbrother/repositories/0015_zenn/workspace/.venv/lib/python3.13/site-packages/rdata/conversion/_conversion.py:900: UserWarning: Missing constructor for R class "tbl". The constructor for class "data.frame" will be used instead.
  warnings.warn(
/loc0/b

Done

coupon_redemptions rda
Done

coupons rda


/loc0/bigbrother/repositories/0015_zenn/workspace/.venv/lib/python3.13/site-packages/rdata/conversion/_conversion.py:900: UserWarning: Missing constructor for R class "spec_tbl_df". The constructor for class "tbl_df" will be used instead.
  warnings.warn(
/loc0/bigbrother/repositories/0015_zenn/workspace/.venv/lib/python3.13/site-packages/rdata/conversion/_conversion.py:900: UserWarning: Missing constructor for R class "tbl_df". The constructor for class "tbl" will be used instead.
  warnings.warn(
/loc0/bigbrother/repositories/0015_zenn/workspace/.venv/lib/python3.13/site-packages/rdata/conversion/_conversion.py:900: UserWarning: Missing constructor for R class "tbl". The constructor for class "data.frame" will be used instead.
  warnings.warn(
/loc0/bigbrother/repositories/0015_zenn/workspace/.venv/lib/python3.13/site-packages/rdata/conversion/_conversion.py:900: UserWarning: Missing constructor for R class "spec_tbl_df". The constructor for class "tbl_df" will be used instead.
  war

Done

demographics rda
Done

products rda
Done



/loc0/bigbrother/repositories/0015_zenn/workspace/.venv/lib/python3.13/site-packages/rdata/conversion/_conversion.py:900: UserWarning: Missing constructor for R class "spec_tbl_df". The constructor for class "tbl_df" will be used instead.
  warnings.warn(
/loc0/bigbrother/repositories/0015_zenn/workspace/.venv/lib/python3.13/site-packages/rdata/conversion/_conversion.py:900: UserWarning: Missing constructor for R class "tbl_df". The constructor for class "tbl" will be used instead.
  warnings.warn(
/loc0/bigbrother/repositories/0015_zenn/workspace/.venv/lib/python3.13/site-packages/rdata/conversion/_conversion.py:900: UserWarning: Missing constructor for R class "tbl". The constructor for class "data.frame" will be used instead.
  warnings.warn(


In [5]:
path_interim = path_project_base / "shared/data/10_interim/completejourney"
os.makedirs(path_interim, exist_ok=True)

In [6]:
for k, v in dfs.items(): 
    v.to_parquet(path_interim / f"{k}.parquet")